# Model 3 — mBERT NER (Multilingual BERT)

**Proyek:** Deteksi Alergen pada Label Pangan Indonesia  
**Mata Kuliah:** COMP6885001 Natural Language Processing — BINUS University

---

## Deskripsi Model

Model 3 menggunakan **mBERT** (`bert-base-multilingual-cased`) — model BERT yang dilatih pada 104 bahasa secara bersamaan — untuk task **Named Entity Recognition (NER)** pada teks bahan makanan.

**Mengapa mBERT?**
- Mendukung 104 bahasa termasuk Bahasa Indonesia dan Inggris
- Lebih umum dari IndoBERT — representasi cross-lingual
- Baseline yang kuat untuk perbandingan dengan model bahasa spesifik

**mBERT vs IndoBERT (Model 4):**
| | mBERT | IndoBERT |
|---|---|---|
| Base model | bert-base-multilingual-cased | indobenchmark/indobert-base-p2 |
| Bahasa pre-training | 104 bahasa | Bahasa Indonesia saja |
| Ukuran | 180M parameter | 124M parameter |
| Fokus ID | Rendah (dibagi 104 bahasa) | Tinggi (full corpus ID) |

**Research Question:** Apakah model yang dikhususkan untuk Bahasa Indonesia (IndoBERT) lebih baik dari model multilingual (mBERT) untuk domain NER label pangan Indonesia?

> **Requires GPU** — Aktifkan Runtime → Change runtime type → GPU (T4/A100)

In [ ]:
# torch sudah pre-install di Colab GPU runtime (dengan CUDA) — jangan reinstall
# hanya install package yang belum ada
!pip install -q datasets pandas pyarrow transformers seqeval numpy

try:
    from seqeval.metrics import f1_score as _sf
    print('seqeval siap')
except ImportError:
    print('seqeval gagal — coba Runtime > Restart and Run All')

import torch
print(f'torch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if not torch.cuda.is_available():
    print('WARNING: GPU tidak aktif — pastikan Runtime > Change runtime type > GPU')

In [ ]:
import torch
print(f'GPU tersedia  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU device    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM          : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: GPU tidak terdeteksi. Training akan sangat lambat di CPU.')

In [ ]:
import re
import json
import os
import random
import numpy as np
import pandas as pd
from collections import Counter
from datasets import load_dataset, Dataset
from transformers import (
    BertTokenizerFast, BertForTokenClassification,
    TrainingArguments, Trainer, DataCollatorForTokenClassification
)
from seqeval.metrics import f1_score, precision_score, recall_score, classification_report as seq_report
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import classification_report, f1_score as sk_f1
from sklearn.metrics import precision_score as sk_p, recall_score as sk_r

random.seed(42)
np.random.seed(42)

## 1. Kamus Alergen & BIO Labeler

In [ ]:
ALLERGEN_CATEGORIES = {
    'GLUTEN': [
        'tepung terigu', 'tepung gandum', 'wheat starch', 'wheat bran', 'wheat germ',
        'malt extract', 'barley malt', 'hydrolyzed wheat protein',
        'tepung', 'terigu', 'gandum', 'gluten', 'barley', 'rye', 'oats',
        'sereal', 'roti', 'mie', 'pasta', 'biskuit', 'cracker', 'couscous',
        'bulgur', 'semolina', 'spelt', 'durum', 'kamut', 'malt', 'farina',
    ],
    'SUSU': [
        'susu sapi', 'milk powder', 'susu bubuk', 'nonfat milk', 'milk protein',
        'susu kental manis', 'whey protein', 'calcium caseinate', 'sodium caseinate',
        'rennet casein', 'milk solids', 'susu evaporasi', 'milk fat',
        'susu', 'keju', 'mentega', 'krim', 'yoghurt', 'butter', 'cream',
        'casein', 'kasein', 'whey', 'laktosa', 'lactose', 'skimmilk',
        'ghee', 'buttermilk', 'lactalbumin', 'lactoglobulin', 'dairy', 'caseinate',
    ],
    'TELUR': [
        'putih telur', 'kuning telur', 'egg white', 'egg yolk', 'egg powder',
        'telur bubuk', 'egg solids', 'egg protein', 'egg lecithin',
        'telur', 'egg', 'ovalbumin', 'ovomucoid', 'ovomucin',
        'lysozyme', 'mayonnaise', 'mayones', 'albumin', 'meringue',
    ],
    'KACANG': [
        'kacang tanah', 'peanut butter', 'kacang pohon', 'kacang mede', 'kacang mete',
        'kacang almond', 'kacang kenari', 'pine nut', 'brazil nut', 'tree nuts',
        'peanut', 'almond', 'cashew', 'walnut', 'pistachio', 'hazelnut',
        'pecan', 'macadamia', 'chestnut', 'groundnut', 'arachis',
    ],
    'KEDELAI': [
        'soy milk', 'susu kedelai', 'soy sauce', 'soy lecithin', 'soy protein',
        'soy isolate', 'soy flour', 'textured soy', 'lesitin kedelai', 'lesitin nabati',
        'soya lecithin',
        'kedelai', 'soy', 'soybean', 'tahu', 'tempe', 'tofu', 'tempeh',
        'edamame', 'miso', 'natto', 'yuba', 'okara',
        'lesitin', 'lecithin',
    ],
    'SEAFOOD': [
        'ikan teri', 'ikan tongkol', 'ikan kembung', 'ikan lele', 'ikan nila',
        'ikan bandeng', 'fish sauce', 'saus ikan', 'shrimp paste', 'fish oil',
        'krill oil', 'anchovy paste', 'saos tiram', 'saus tiram', 'oyster sauce',
        'ikan', 'tuna', 'salmon', 'sarden', 'makarel', 'cod', 'herring',
        'anchovy', 'udang', 'kerang', 'prawn', 'shrimp', 'cumi', 'sotong',
        'kepiting', 'crab', 'lobster', 'mussel', 'oyster', 'scallop',
        'abalone', 'terasi',
    ],
    'WIJEN': [
        'sesame oil', 'minyak wijen', 'biji wijen',
        'wijen', 'sesame', 'tahini',
    ],
    'SULFITE': [
        'sodium sulfite', 'potassium metabisulfite', 'sulfur dioxide',
        'natrium sulfit', 'kalium metabisulfit',
        'sulfite', 'sulfit',
    ],
}

ALL_CATEGORIES = list(ALLERGEN_CATEGORIES.keys())

def build_phrase_list(categories):
    phrases = []
    for cat, words in categories.items():
        for word in words:
            phrases.append((word.lower(), cat))
    phrases.sort(key=lambda x: len(x[0].split()), reverse=True)
    return phrases

PHRASE_LIST = build_phrase_list(ALLERGEN_CATEGORIES)

def tokenize_text(text):
    return re.findall(r'\w+|[^\w\s]', text.lower())

def bio_labeler(text):
    tokens = tokenize_text(text)
    labels = ['O'] * len(tokens)
    i = 0
    while i < len(tokens):
        matched = False
        for phrase, cat in PHRASE_LIST:
            phrase_tokens = tokenize_text(phrase)
            plen = len(phrase_tokens)
            if tokens[i:i + plen] == phrase_tokens:
                if all(labels[j] == 'O' for j in range(i, i + plen)):
                    labels[i] = f'B-{cat}'
                    for j in range(i + 1, i + plen):
                        labels[j] = f'I-{cat}'
                    i += plen
                    matched = True
                    break
        if not matched:
            i += 1
    return list(zip(tokens, labels))

print(f'ALLERGEN_CATEGORIES siap. {len(ALL_CATEGORIES)} kategori.')

## 2. Data Loading & Labeling

Sama dengan pipeline NLP RM — Open Food Facts (EN+ID) + 500 data sintetis.

In [ ]:
# ==========================================
# KONFIGURASI
TOTAL_DATA  = 3000
N_SYNTHETIC = 500
# ==========================================

print('Streaming Open Food Facts...')
dataset = load_dataset('openfoodfacts/product-database', split='food', streaming=True)

sample_data = []
for example in dataset:
    lang = example.get('lang', '')
    if lang not in ('en', 'id'):
        continue
    raw = example.get('ingredients_text')
    if not raw:
        continue
    text = raw[0].get('text', '') if (isinstance(raw, list) and isinstance(raw[0], dict)) else str(raw)
    if not text or text == 'nan' or len(text) < 5:
        continue
    sample_data.append({'product_name': example.get('product_name', 'Unknown'), 'ingredients_text': text})
    if len(sample_data) >= TOTAL_DATA:
        break

df_off = pd.DataFrame(sample_data)

# Data sintetis
NON_ALLERGEN_WORDS = [
    'garam', 'air', 'gula', 'minyak sawit', 'pati', 'vitamin c',
    'kalium sorbat', 'asam sitrat', 'pewarna makanan', 'antioksidan',
    'pengatur keasaman', 'natrium klorida', 'glukosa', 'asam laktat',
    'karagenan', 'agar', 'pektin', 'karamel', 'natrium bikarbonat',
    'perisa', 'ekstrak vanili', 'kunyit',
]
TEMPLATES_ID = [
    'Komposisi: {bahan}.', 'Bahan: {bahan}.', 'Mengandung: {bahan}.', '{bahan}.',
]

random.seed(42)
all_words = [(w, cat) for cat, words in ALLERGEN_CATEGORIES.items() for w in words]
synthetic = []
for _ in range(N_SYNTHETIC):
    n_alr = random.randint(1, 3)
    n_non = random.randint(2, 5)
    selected_alr = random.sample(all_words, min(n_alr, len(all_words)))
    selected_non = random.sample(NON_ALLERGEN_WORDS, min(n_non, len(NON_ALLERGEN_WORDS)))
    ingredients  = [w for w, _ in selected_alr] + selected_non
    random.shuffle(ingredients)
    text = random.choice(TEMPLATES_ID).format(bahan=', '.join(ingredients))
    synthetic.append({'product_name': 'Synthetic-ID', 'ingredients_text': text})

df_synthetic = pd.DataFrame(synthetic)
df_combined  = pd.concat([df_off, df_synthetic], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
print(f'OFF: {len(df_off)} | Sintetis: {N_SYNTHETIC} | Total: {len(df_combined)}')

In [ ]:
labeled_dataset = []
for _, row in df_combined.iterrows():
    text = str(row['ingredients_text'])
    if not text or text == 'nan' or len(text) < 3:
        continue
    bio = bio_labeler(text)
    if bio:
        labeled_dataset.append({'clean_ingredients': text, 'tokens_and_tags': bio})

all_labels = [tag for item in labeled_dataset for _, tag in item['tokens_and_tags']]
label_counts = Counter(all_labels)
print(f'Total sampel: {len(labeled_dataset)}, total token: {len(all_labels):,}')
print('Distribusi label allergen:')
for lbl, cnt in sorted(label_counts.items()):
    if lbl != 'O':
        print(f'  {lbl:<18}: {cnt:>6}')

## 3. mBERT Tokenizer & Label Setup

**Perbedaan utama dari IndoBERT:**  
`bert-base-multilingual-cased` → `BertTokenizerFast`

mBERT menggunakan **cased** tokenizer (huruf besar/kecil dibedakan), sedangkan IndoBERT menggunakan uncased. Ini relevan karena label produk sering menggunakan huruf kapital.

In [ ]:
# ==========================================
# PERBEDAAN DARI INDOBERT: base model berbeda
MODEL_NAME = 'bert-base-multilingual-cased'
# ==========================================

tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)

label_list = ['O']
for cat in ALLERGEN_CATEGORIES.keys():
    label_list.append(f'B-{cat}')
    label_list.append(f'I-{cat}')

label2id = {l: i for i, l in enumerate(label_list)}
id2label  = {i: l for i, l in enumerate(label_list)}

print(f'Model      : {MODEL_NAME}')
print(f'Vocab size : {tokenizer.vocab_size:,}')
print(f'Labels     : {len(label_list)}')

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples['tokens'],
        truncation=True,
        max_length=512,
        is_split_into_words=True,
    )
    all_label_ids = []
    for i, label_seq in enumerate(examples['ner_tags']):
        word_ids = tokenized.word_ids(batch_index=i)
        prev_word_id = None
        label_ids = []
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != prev_word_id:
                label_ids.append(label2id.get(label_seq[word_id], 0))
            else:
                label_ids.append(-100)
            prev_word_id = word_id
        all_label_ids.append(label_ids)
    tokenized['labels'] = all_label_ids
    return tokenized

formatted = {
    'tokens'  : [[t for t, _ in item['tokens_and_tags']] for item in labeled_dataset],
    'ner_tags': [[l for _, l in item['tokens_and_tags']] for item in labeled_dataset],
}

full_dataset = Dataset.from_dict(formatted)
tokenized_dataset = full_dataset.map(tokenize_and_align_labels, batched=True)

TEST_SIZE = 300
split      = tokenized_dataset.train_test_split(test_size=TEST_SIZE, seed=42)
train_data = split['train']
test_data  = split['test']

print(f'Train: {len(train_data)} | Test: {len(test_data)}')

## 4. Training mBERT NER

Setup training **identik dengan IndoBERT** (10 epoch, lr=2e-5, batch=8) untuk perbandingan fair.

In [ ]:
def compute_metrics(p):
    logits, labels = p
    preds = np.argmax(logits, axis=2)
    true_preds = [
        [id2label[pred] for pred, lbl in zip(pr, lr) if lbl != -100]
        for pr, lr in zip(preds, labels)
    ]
    true_labels = [
        [id2label[lbl] for pred, lbl in zip(pr, lr) if lbl != -100]
        for pr, lr in zip(preds, labels)
    ]
    return {
        'precision': precision_score(true_labels, true_preds),
        'recall'   : recall_score(true_labels, true_preds),
        'f1'       : f1_score(true_labels, true_preds),
    }

In [ ]:
# ==========================================
# KONFIGURASI TRAINING — sama dengan IndoBERT
NUM_EPOCHS    = 10
BATCH_SIZE    = 8
LEARNING_RATE = 2e-5
OUTPUT_DIR    = './results_mbert'
# ==========================================

model = BertForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='epoch',
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    push_to_hub=False,
    report_to='none',
)

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

print(f'Training mBERT NER: {NUM_EPOCHS} epoch, lr={LEARNING_RATE}, batch={BATCH_SIZE}')
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')
trainer.train()
print('Training selesai!')

In [ ]:
print('=== Evaluasi Silver Label (Test Split) ===')
pred_output  = trainer.predict(test_data)
preds_eval   = np.argmax(pred_output.predictions, axis=2)
labels_eval  = pred_output.label_ids

true_preds = [
    [id2label[p] for p, l in zip(pr, lr) if l != -100]
    for pr, lr in zip(preds_eval, labels_eval)
]
true_labels = [
    [id2label[l] for p, l in zip(pr, lr) if l != -100]
    for pr, lr in zip(preds_eval, labels_eval)
]

print(seq_report(true_labels, true_preds))
silver_f1 = f1_score(true_labels, true_preds)
print(f'Silver Micro F1: {silver_f1:.4f}')

## 5. Inferensi & Evaluasi Gold Label

In [ ]:
def predict_allergens_mbert(text, model, tokenizer, id2label):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model.to(device)
    model.eval()
    inputs = tokenizer(text.lower(), return_tensors='pt', truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    pred_ids = torch.argmax(outputs.logits, dim=2)[0].tolist()
    tokens   = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    results, cur_word, cur_label = [], '', 'O'
    for token, pid in zip(tokens, pred_ids):
        if token in ['[CLS]', '[SEP]', '[PAD]']:
            continue
        if token.startswith('##'):
            cur_word += token[2:]
        else:
            if cur_word:
                results.append((cur_word, cur_label))
            cur_word, cur_label = token, id2label[pid]
    if cur_word:
        results.append((cur_word, cur_label))
    return results

def predict_categories_mbert(text):
    results = predict_allergens_mbert(text, model, tokenizer, id2label)
    return sorted({label[2:] for _, label in results if label.startswith('B-')})

# Demo
DEMO_TEXTS = [
    'Komposisi: tepung terigu, gula, susu bubuk, kacang tanah, perisa alami.',
    'Bahan: tuna, shrimp paste, soy sauce, sesame oil, garam.',
    'Bahan: lesitin nabati, minyak sawit, coklat, susu kental manis, telur.',
]
print('=== Demo Inferensi Model 3 (mBERT NER) ===')
for text in DEMO_TEXTS:
    cats = predict_categories_mbert(text)
    print(f'Input    : {text}')
    print(f'Prediksi : {cats}')
    print()

In [ ]:
EVAL_FILE = 'evaluation_dataset.json'

if not os.path.exists(EVAL_FILE):
    print(f'{EVAL_FILE} tidak ditemukan — evaluasi gold label dilewati.')
    M3_RESULTS = None
else:
    with open(EVAL_FILE, 'r', encoding='utf-8') as f:
        eval_data = json.load(f)

    y_true_raw, y_pred_raw = [], []
    for item in eval_data:
        pred = predict_categories_mbert(item['ingredient_text'])
        y_pred_raw.append(pred)
        y_true_raw.append(item['ground_truth'])

    mlb_eval = MultiLabelBinarizer(classes=ALL_CATEGORIES)
    Y_true   = mlb_eval.fit_transform(y_true_raw)
    Y_pred   = mlb_eval.transform(y_pred_raw)

    print('=== MODEL 3: mBERT NER (Gold Label Evaluation) ===')
    print(classification_report(Y_true, Y_pred, target_names=ALL_CATEGORIES, zero_division=0))

    micro_p = sk_p(Y_true, Y_pred, average='micro', zero_division=0)
    micro_r = sk_r(Y_true, Y_pred, average='micro', zero_division=0)
    micro_f = sk_f1(Y_true, Y_pred, average='micro', zero_division=0)
    macro_f = sk_f1(Y_true, Y_pred, average='macro', zero_division=0)

    print(f'Micro Precision : {micro_p:.4f}')
    print(f'Micro Recall    : {micro_r:.4f}')
    print(f'Micro F1        : {micro_f:.4f}')
    print(f'Macro F1        : {macro_f:.4f}')

    M3_RESULTS = {
        'model': 'mBERT NER',
        'silver_micro_f1': round(silver_f1, 4),
        'micro_precision': round(micro_p, 4),
        'micro_recall': round(micro_r, 4),
        'micro_f1': round(micro_f, 4),
        'macro_f1': round(macro_f, 4),
    }
    print(f'\nM3_RESULTS = {M3_RESULTS}')

## 6. Simpan Model

In [ ]:
import shutil

MODEL_SAVE_PATH = './model_mbert_alergen'
trainer.save_model(MODEL_SAVE_PATH)
tokenizer.save_pretrained(MODEL_SAVE_PATH)
shutil.make_archive('model_mbert_alergen', 'zip', MODEL_SAVE_PATH)

try:
    from google.colab import files
    files.download('model_mbert_alergen.zip')
except ImportError:
    pass

print(f'Model mBERT tersimpan di: {MODEL_SAVE_PATH}')

## 7. Analisis: mBERT vs IndoBERT

### Hipotesis
IndoBERT (Model 4) diharapkan lebih baik dari mBERT (Model 3) karena:
1. IndoBERT dilatih **hanya** pada corpus Bahasa Indonesia → representasi yang lebih kaya
2. mBERT membagi kapasitas model untuk 104 bahasa → representasi per bahasa lebih tipis
3. Label pangan Indonesia sering menggunakan bahasa campuran EN+ID → mBERT mungkin menangani ini lebih baik

### Catatan Evaluasi
- **Silver F1** (test split dari data yang sama): Kemungkinan besar keduanya tinggi (>0.95) — silver label circular evaluation
- **Gold F1** (50 produk real-world): Ukuran performa yang lebih jujur — di sini perbedaan mBERT vs IndoBERT akan terlihat

### Implikasi untuk Praktik
- Jika IndoBERT >> mBERT pada gold F1: Penggunaan model bahasa spesifik dianjurkan untuk domain pangan Indonesia
- Jika mBERT ≈ IndoBERT: mBERT bisa jadi pilihan karena lebih umum dan mudah diakses

In [ ]:
print('=== PERBANDINGAN MODEL 3 (mBERT) vs MODEL 4 (IndoBERT) ===')
print()
print(f'{"Model":<35} {"Silver F1":>10} {"Gold F1":>10}')
print('-' * 60)

m3_silver = M3_RESULTS.get('silver_micro_f1', 'N/A') if M3_RESULTS else 'N/A'
m3_gold   = M3_RESULTS.get('micro_f1', 'N/A') if M3_RESULTS else 'N/A'
print(f'{"mBERT NER (Model 3)":<35} {str(m3_silver):>10} {str(m3_gold):>10}')

# IndoBERT hasil dari Model 4 notebook (isi manual setelah run Model 4)
INDOBERT_SILVER_F1 = 0.9946  # dari training RM
INDOBERT_GOLD_F1   = 0.7490  # dari real-world eval RM
print(f'{"IndoBERT NER (Model 4)":<35} {INDOBERT_SILVER_F1:>10.4f} {INDOBERT_GOLD_F1:>10.4f}')
print()
print('Catatan: Silver F1 diukur pada test split dari data yang sama (silver label bias).')
print('         Gold F1 diukur pada 50 produk Indonesia yang dianotasi manual.')